In [1]:
import pandas as pd
import duckdb

#from juptils import lefty
#from juptils import show_all
#from juptils import save_csv

# Make monthly sales summary.

## load parts.csv 

In [2]:
parts = pd.read_csv("./csv/parts.csv")
parts['SQFT'] = parts['SQFT'].astype('Int64')
parts

,PART_ID,PROD,PART_TYPE,SQFT
0,tin-clip,tin-roof,nan,10
1,tin-decoration,tin-roof,nan,
2,gold-clip,golf-roof,nan,50
3,gold-decoration,golf-roof,nan,
4,glass-clip,glass-roof,new,100
5,glass-decoration,glass-roof,new,
6,roof-polish,nan,nan,


## Load sales.csv 

- Test data generated by script Grok made.
- 2 suppliers with 2 licensees each, plus another licensee, who buys from both suppliers.
- Date range 3/2023 to 1/2026

In [3]:
sales = pd.read_csv("./csv/sales-grok.csv", parse_dates=['DATE'])
sales['AMT'] = sales['AMT'].astype('int64')
sales['QTY'] = sales['QTY'].astype('int64')
sales

,DATE,SUPPLIER,LICENSEE,PO,PART_ID,QTY,AMT
0,2023-03-08 00:00:00,nippon-metal,ninja-roofing,PO0001,roof-polish,57,11100
1,2023-03-08 00:00:00,nippon-metal,ninja-roofing,PO0001,glass-decoration,114,86100
2,2023-03-08 00:00:00,nippon-metal,ninja-roofing,PO0001,gold-clip,156,89500
3,2023-03-08 00:00:00,nippon-metal,ninja-roofing,PO0001,glass-clip,109,128600
4,2023-03-08 00:00:00,nippon-metal,ninja-roofing,PO0001,gold-decoration,165,64400
5,2023-03-08 00:00:00,nippon-metal,ninja-roofing,PO0001,tin-decoration,44,3800
6,2023-03-08 00:00:00,nippon-metal,ninja-roofing,PO0001,tin-clip,176,16000
7,2023-03-04 00:00:00,nippon-metal,rice-roofers,PO0002,gold-clip,78,35300
8,2023-03-04 00:00:00,nippon-metal,rice-roofers,PO0002,tin-decoration,95,8900
9,2023-03-04 00:00:00,nippon-metal,rice-roofers,PO0002,gold-decoration,119,57800


## Monthly sales per supplier, licensee, part.  
## with cummulative monthly totals.

In [5]:
duckdb.query("DROP VIEW IF EXISTS ytd_sales_2");
duckdb.query("DROP VIEW IF EXISTS ytd_sales");

duckdb.query("""
CREATE VIEW ytd_sales AS
SELECT 
    strftime('%Y-%m', date) AS month,
    supplier,
    licensee,
    part_id,
    CAST(SUM(qty) AS INT64) AS qty,
    CAST(SUM(amt) AS INT64) AS month_amt,
    CAST(
        SUM(month_amt) OVER (
            PARTITION BY SUBSTR(month, 1, 4), supplier, licensee, part_id 
            ORDER BY month
        ) AS INT64
    ) AS ytd_amt
FROM sales
GROUP BY month, supplier, licensee, part_id
ORDER BY month, supplier, licensee, part_id;
""")

ytd_df = duckdb.query("select * from ytd_sales").df()
ytd_df.to_csv('./csv/ytd_sales.csv', index=False)


ytd_df.lefty()

,month,SUPPLIER,LICENSEE,PART_ID,qty,month_amt,ytd_amt
0,2023-03,nippon-metal,ninja-roofing,glass-clip,109,128600,128600
1,2023-03,nippon-metal,ninja-roofing,glass-decoration,114,86100,86100
2,2023-03,nippon-metal,ninja-roofing,gold-clip,156,89500,89500
3,2023-03,nippon-metal,ninja-roofing,gold-decoration,165,64400,64400
4,2023-03,nippon-metal,ninja-roofing,roof-polish,57,11100,11100
5,2023-03,nippon-metal,ninja-roofing,tin-clip,176,16000,16000
6,2023-03,nippon-metal,ninja-roofing,tin-decoration,44,3800,3800
7,2023-03,nippon-metal,rice-roofers,glass-clip,158,186200,186200
8,2023-03,nippon-metal,rice-roofers,gold-clip,78,35300,35300
9,2023-03,nippon-metal,rice-roofers,gold-decoration,119,57800,57800


In [8]:
xf = duckdb.query("""
select * 
from ytd_sales
where 
    month between '2023-03' and '2023-06'
    and
    supplier = 'nippon-metal'
    and 
    licensee = 'ninja-roofing'
    and 
    part_id = 'glass-clip'
    
""").df()

xf

,month,SUPPLIER,LICENSEE,PART_ID,qty,month_amt,ytd_amt
0,2023-03,nippon-metal,ninja-roofing,glass-clip,109,128600,128600
1,2023-05,nippon-metal,ninja-roofing,glass-clip,101,103900,232500
2,2023-06,nippon-metal,ninja-roofing,glass-clip,225,231600,464100
